# 07 (FIXED) — Training Time, Inference Time, Memory & Scalability Benchmark

**Addresses:** R1-Q10.

**Fixes vs. the original `07_Training_Inference_Memory_Benchmark.ipynb`:**
1. **Wrong architecture** — the original redefined a throwaway `Generator` with 2 conv + 2 transposed-conv layers and *no residual blocks*, instead of importing the real `model.py` `Generator` (6 residual blocks, per the manuscript). Every number it reported (params, memory, timing) described a different, much smaller network than EyeGAN. This version imports the real `Generator`/`Discriminator` from `model.py`.
2. **No GPU warm-up before timing** — the original looped `[64, 128]` and started timing immediately; the first resolution absorbs one-time CUDA/cuDNN kernel-selection overhead, which is why 64x64 (434s/epoch) came out *slower* than 128x128 (98s/epoch) in the original output. This version runs a few untimed warm-up iterations per resolution before starting the clock, and resets peak-memory stats after warm-up.
3. **Undefined variable** — the original's scalability-probe cell referenced `result_64`, which was never assigned anywhere, so it would raise `NameError` on a fresh run. Fixed by accumulating results in a single list as the loop runs.
4. **Cell-order bug** — the original called `benchmark_inference()` in an earlier cell than the one that *defines* it, so the notebook only worked because of stale state from being run out of order in a live Colab session. All functions are defined before first use here.

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSE720/code')
import os, time
import torch
import pandas as pd
from config import Config

# Import the REAL architecture -- do not redefine it locally.
# If your model.py names the discriminator class differently, adjust this import.
from model import Generator, Discriminator

cfg = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

os.makedirs(cfg.benchmark_dir if hasattr(cfg, 'benchmark_dir')
            else os.path.join(cfg.eval_dir, 'benchmarks'), exist_ok=True)
BENCH_DIR = cfg.benchmark_dir if hasattr(cfg, 'benchmark_dir') else os.path.join(cfg.eval_dir, 'benchmarks')

Mounted at /content/drive
Device: cuda


## 1. Training-time & memory benchmark function (real architecture, warm-up included)

In [2]:
def benchmark_training(img_size, batch_size, num_domains=5, warmup_steps=5, timed_steps=30):
    """
    Times `timed_steps` forward+backward steps at a fixed (img_size, batch_size),
    after `warmup_steps` untimed steps to absorb one-time CUDA/cuDNN kernel-selection
    overhead. Uses random dummy data (not the real dataset) since only architecture and
    input size affect timing/memory, not image content -- this also means it works at
    resolutions (128, 256) where no trained data actually exists.
    """
    torch.cuda.empty_cache()

    G = Generator(img_size, num_domains=num_domains).to(device)
    D = Discriminator(img_size, num_domains=num_domains).to(device)
    g_opt = torch.optim.Adam(G.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
    d_opt = torch.optim.Adam(D.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))

    def one_step():
        real = torch.randn(batch_size, 3, img_size, img_size, device=device)
        src_labels = torch.randint(0, num_domains, (batch_size,), device=device)
        tgt_labels = torch.randint(0, num_domains, (batch_size,), device=device)

        # --- Discriminator step ---
        d_opt.zero_grad()
        real_src, real_cls = D(real)
        fake = G(real, tgt_labels)
        fake_src, _ = D(fake.detach())
        d_loss = (real_src.mean() - fake_src.mean()) \
                 + torch.nn.functional.cross_entropy(real_cls, src_labels)
        d_loss.backward()
        d_opt.step()

        # --- Generator step (adv + cls + cycle + identity, matching the paper's 4 loss terms) ---
        g_opt.zero_grad()
        fake = G(real, tgt_labels)
        fake_src, fake_cls = D(fake)
        recon = G(fake, src_labels)
        identity = G(real, src_labels)
        g_loss = (-fake_src.mean()
                  + torch.nn.functional.cross_entropy(fake_cls, tgt_labels)
                  + cfg.lambda_cycle * torch.nn.functional.l1_loss(recon, real)
                  + cfg.lambda_identity * torch.nn.functional.l1_loss(identity, real))
        g_loss.backward()
        g_opt.step()

    # Warm-up: absorb cuDNN algorithm-selection / memory-allocator overhead, untimed.
    for _ in range(warmup_steps):
        one_step()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    for _ in range(timed_steps):
        one_step()
    torch.cuda.synchronize()
    elapsed = time.time() - t0

    per_step_time = elapsed / timed_steps
    # Scale a per-step time up to a per-epoch estimate using the real training-set size.
    steps_per_epoch = max(1, cfg.n_train_images // batch_size) if hasattr(cfg, 'n_train_images') else None

    total_params = sum(p.numel() for p in G.parameters())
    peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    del G, D, g_opt, d_opt
    torch.cuda.empty_cache()

    result = {
        'img_size': img_size,
        'batch_size': batch_size,
        'mean_step_time_sec': per_step_time,
        'peak_gpu_memory_MB': peak_mem_mb,
        'generator_params': total_params,
    }
    if steps_per_epoch is not None:
        result['estimated_epoch_time_sec'] = per_step_time * steps_per_epoch
        result['estimated_100_epoch_hours'] = per_step_time * steps_per_epoch * 100 / 3600
    return result

## 2. Inference-latency benchmark function (real architecture, warm-up included)

For the native 64x64 resolution, this loads your actual trained checkpoint so the reported
latency reflects the real deployed model; for the scalability probe at 128/256 (where no
checkpoint exists at that size) it uses freshly-initialized weights of the real architecture,
since weight values don't affect latency, only architecture + input size do.

In [3]:
def benchmark_inference(img_size, num_domains=5, checkpoint_path=None, warmup_steps=10, n_trials=100):
    G = Generator(img_size, num_domains=num_domains).to(device)
    if checkpoint_path is not None and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        G.load_state_dict(ckpt['G_state_dict'])
        print(f'  Loaded real trained weights from {checkpoint_path}')
    else:
        print(f'  No checkpoint at this size -- using freshly-initialized weights '
              f'(architecture/timing only, not representative of trained-model quality)')
    G.eval()

    dummy = torch.randn(1, 3, img_size, img_size, device=device)
    label = torch.tensor([0], device=device)

    with torch.no_grad():
        for _ in range(warmup_steps):
            G(dummy, label)
        torch.cuda.synchronize()

        t0 = time.time()
        for _ in range(n_trials):
            G(dummy, label)
        torch.cuda.synchronize()
        single_ms = (time.time() - t0) / n_trials * 1000

        throughputs = {}
        for bs in [1, 8, 32]:
            batch = torch.randn(bs, 3, img_size, img_size, device=device)
            labels = torch.zeros(bs, dtype=torch.long, device=device)
            for _ in range(5):
                G(batch, labels)
            torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(20):
                G(batch, labels)
            torch.cuda.synchronize()
            throughputs[f'batch_{bs}_imgs_per_sec'] = bs * 20 / (time.time() - t0)

    del G
    torch.cuda.empty_cache()
    return {'img_size': img_size, 'single_image_latency_ms': single_ms, **throughputs}

## 3. Run the benchmark at the native resolution + the scalability probe

One consolidated loop (the original had this split across two separate, inconsistent cells).

In [4]:
REAL_CHECKPOINT = os.path.join(cfg.checkpoint_dir, 'final_model.pth') if hasattr(cfg, 'checkpoint_dir') else None

# (img_size, batch_size); native config first, then scalability probe at higher res.
configs = [(64, 8), (128, 4), (256, 2)]

training_results = []
inference_results = []

for img_size, batch_size in configs:
    print(f'\n--- Profiling {img_size}x{img_size}, batch={batch_size} ---')
    try:
        r_train = benchmark_training(img_size=img_size, batch_size=batch_size)
        training_results.append(r_train)
        print('  Training:', r_train)

        ckpt_for_this_size = REAL_CHECKPOINT if img_size == cfg.img_size else None
        r_infer = benchmark_inference(img_size=img_size, checkpoint_path=ckpt_for_this_size)
        inference_results.append(r_infer)
        print('  Inference:', r_infer)

    except torch.cuda.OutOfMemoryError:
        print(f'  {img_size}x{img_size} @ batch={batch_size}: OUT OF MEMORY -- '
              f'this itself is a valid, reportable data point for the scalability discussion.')
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'  {img_size}x{img_size}: failed with {type(e).__name__}: {e}')
        torch.cuda.empty_cache()

train_df = pd.DataFrame(training_results)
infer_df = pd.DataFrame(inference_results)

print('\n=== Training / memory scalability ===')
print(train_df.round(4).to_string(index=False))
print('\n=== Inference latency / throughput ===')
print(infer_df.round(4).to_string(index=False))

train_csv = os.path.join(BENCH_DIR, 'training_scalability_FIXED.csv')
infer_csv = os.path.join(BENCH_DIR, 'inference_scalability_FIXED.csv')
train_df.to_csv(train_csv, index=False)
infer_df.to_csv(infer_csv, index=False)
print(f'\nSaved to {train_csv} and {infer_csv}')
print('\nUse the img_size == cfg.img_size (native, 64x64) row for the Results section --')
print('the 128/256 rows are the scalability probe for the Limitations/Discussion.')


--- Profiling 64x64, batch=8 ---
  Training: {'img_size': 64, 'batch_size': 8, 'mean_step_time_sec': 0.04300088882446289, 'peak_gpu_memory_MB': 200.24267578125, 'generator_params': 2048675}
  Loaded real trained weights from /content/drive/MyDrive/CSE720/stargan_checkpoints/final_model.pth
  Inference: {'img_size': 64, 'single_image_latency_ms': 2.123558521270752, 'batch_1_imgs_per_sec': 518.5547292744593, 'batch_8_imgs_per_sec': 2138.1919211872887, 'batch_32_imgs_per_sec': 2461.051089403017}

--- Profiling 128x128, batch=4 ---
  Training: {'img_size': 128, 'batch_size': 4, 'mean_step_time_sec': 0.07610525290171305, 'peak_gpu_memory_MB': 366.75, 'generator_params': 3633827}
  No checkpoint at this size -- using freshly-initialized weights (architecture/timing only, not representative of trained-model quality)
  Inference: {'img_size': 128, 'single_image_latency_ms': 2.1413040161132812, 'batch_1_imgs_per_sec': 494.3519025982827, 'batch_8_imgs_per_sec': 553.2018738783892, 'batch_32_imgs

## Next
Send me the printed `train_df` / `infer_df` output (or the two CSVs) and I'll correct the
"Computational cost and scalability" numbers in the manuscript, which currently still have
the wrong-architecture figures (0.59M params, 434s/epoch, 193MB, 0.53ms) from the original
buggy notebook.